<a href="https://colab.research.google.com/github/prometheus404/NLP_proj/blob/master/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NLP project

In [2]:
%pip install llama-cpp-python==0.2.90 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu122


In [3]:
from llama_cpp import Llama
from tqdm import tqdm
#from transformers import AutoTokenizer, pipeline, BitsAndBytesConfig
import requests
from collections import defaultdict
import json


In [4]:
# Load the model
mistral = Llama.from_pretrained(repo_id="bartowski/Meta-Llama-3.1-8B-Instruct-GGUF", # repository name
                            filename="Meta-Llama-3.1-8B-Instruct-Q8_0.gguf", # model file
                            n_gpu_layers=-1, # use all GPU layers
                            n_ctx=32768, # context size
                            flash_attn=True, # use flash attention
                            chat_format="llama-3", # chat format
                            verbose=False)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
rulebook = requests.get('https://raw.githubusercontent.com/prometheus404/NLP_proj/refs/heads/master/rules/texts/dominion.txt').text
rulebook[:100]

'# Dominion\nYou are a monarch, like your parents before you - a ruler of a small pleasant kingdom of '

In [ ]:
# Check that input is inside context window
#tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1")

#tokens = tokenizer.encode(rulebook)
#print(len(tokens))

In [6]:
def generate_message(prompt, rulebook):
    return [
        {
                "role": "system",
                "content": prompt,
            },
            {
                "role": "user",
                "content": "Here is the rulebook:\n"+rulebook,
            },
    ]

In [21]:
import torch
def multiple_model_test(models, prompts, games, iterations, test_name):
    outputs = defaultdict(dict)

    for model_name,model,game,prompt_name,prompt,it in tqdm([(mn,m,f,pn,p,it) for f in game_names for (pn,p) in prompts for (mn,m) in models for it in range(iterations)]):
        rulebook = requests.get('https://raw.githubusercontent.com/prometheus404/NLP_proj/refs/heads/master/rules/texts/'+game+'.txt').text
        out = model.create_chat_completion(generate_message(prompt, rulebook), temperature=0.7)
        outputs[model_name][game+'-'+prompt_name+'-'+str(it)] = out['choices'][0]['message']['content']
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


    with open(f'{test_name}.out','w') as f:
        json.dump(dict(outputs),f)

    return outputs

# Rule extraction
1. Give the model a rulebook and prompt it to explain the game in simple, conversational terms to a child or other audiences. -> tree decomposition to test how well the model did
2. Test the ability of the model to find analogies of rules (?)
3. Test the ability to extract if-then rules (?)
4. Organize the rules of into a hierarchy: top-level objectives, mid-level phases, low-level actions. (?) (look into the paper)

In [22]:
prompts = [("kid", """You are a friendly tutor explaining board games to a 7‑year‑old. Summarize the game in plain language, using short sentences and with fun tone. Include:
                - Goal of the game
                - How a player wins
                - What a turn looks like
                - Exceptions to standard rules

                The user will give you a text file with the rulebook you need to explain.
                the output should not be too long. All rules must be present in your explanation"""),
           ("analogies", """The user will give you a text file with the rulebook you need to explain
           the output should not be too long. All rules must be present in your explanation.
           Explain the rules to a child by comparing it to something they already know (e.g., “like a treasure hunt” or “like building a LEGO city”).
           se the rulebook to keep the analogy accurate, and end with a one‑sentence “what you try to achieve” statement.\n"""),
]
game_names = [ 'dominion','7_wonders', 'catan', 'power_grid','ticket_to_ride',]
models = [("mistral", mistral)]
iterations = 1

multiple_model_test(models, prompts, game_names, iterations, 'extraction')

100%|██████████| 10/10 [03:07<00:00, 18.72s/it]


defaultdict(dict,
            {'mistral': {'dominion-kid-0': 'Hey there, kiddo! Let\'s talk about the game Dominion!\n\n**What\'s the goal of the game?**\nThe goal of the game is to build the best deck of cards, called a Dominion. You want to have the most cards in your deck when the game ends.\n\n**How do you win the game?**\nYou win the game by having the most cards in your deck when the game ends. The game ends when three or more of the Supply piles are empty, or the Province pile is empty.\n\n**What\'s a turn like?**\nA turn has three phases: Action, Buy, and Clean-up.\n\n1. **Action Phase**: You can play one Action card from your hand. An Action card is a card that says "Action" on the bottom and has a white banner. You can follow the instructions on the card to do things like gain cards, trash cards, or draw cards.\n2. **Buy Phase**: You can play Treasure cards from your hand to get coins, and then use those coins to buy one card from the Supply.\n3. **Clean-up Phase**: You put a

## Error detection

 Each rulebook is edited by inserting a set of 5 errors each of increasing difficulty:
 - level 1 -> **unsolvable** mechanic that uses piece not presents in the game (every time you play a card you can steal a warrior token from every opponent but there is no way to obtain a warrior token)
- level 2 -> **unsolvable** (in one line states you can draw two cards, in another one that you can draw only one)
- level 3 -> **incoherent** and hardlocks the game (you cannot play train if you do not have train on the map but on another line clearly states you start with an empty map)
- level 4 -> **coherent** but obviously gamebreaking (draw infinite cards each turn)
- level 5 -> **coherent** but very unbalanced (the first player can play two turns)

In [ ]:
prompts = ["""test"""]
game_names = ['ticket_to_ride', 'dominion', 'catan', 'power_grid']
models = [llm]
iterations = 5

multiple_model_test(models, prompts, game_names, iterations, 'error_detection')

# Game classification
Give the model a rulebook and ask it to classify the mechanics, evaluate the complexity, suggests the perfect number of players and estimate the duration